# NEB-TS Search: Command Assembly & Submission

Assembles shell commands for MLFF NEB transition-state searches, writes `.sh` submit scripts, and prints paste-ready commands for interactive GPU testing.

**Modes:**
- `quick` — NEB only (no climbing image, no Sella). Fastest screening for many theozymes.
- `standard` — NEB + climbing-image NEB. Good TS estimate without Sella overhead.
- `full` — NEB + CI-NEB + Sella refinement + frequency validation. Publication quality.

In [ ]:
import os, sys, glob, math, stat
from pathlib import Path

---
## 1. Paths & Environment

In [ ]:
### PATHS ###
HOME_DIR    = os.path.expanduser('~') + '/'
PROJECT_DIR = f'{HOME_DIR}dft_and_QM_stuff/mlffs_and_mace/neb_TS_search_seth/'
DATA_DIR    = '/net/scratch/lschaaf/projects/seth/data/pte_studies/'

# >> subdirectories
CMDS_DIR   = os.path.join(PROJECT_DIR, 'cmds/')   ; os.makedirs(CMDS_DIR, exist_ok=True)
SUBMIT_DIR = os.path.join(PROJECT_DIR, 'submit/')  ; os.makedirs(SUBMIT_DIR, exist_ok=True)
LOGS_DIR   = os.path.join(PROJECT_DIR, 'logs/')    ; os.makedirs(LOGS_DIR, exist_ok=True)

### ENVIRONMENT ###
# universal.sif has mace 0.3.15 + e3nn 0.4.4 + torch 2.11 + ASE + biotite
# .local_pkgs adds sella + jax on top (only missing pieces)
CONTAINER  = '/net/software/containers/universal.sif'
LOCAL_PKGS = os.path.join(PROJECT_DIR, '.local_pkgs')
SCRIPT     = os.path.join(PROJECT_DIR, 'run_neb_ts.py')

print(f'Project:   {PROJECT_DIR}')
print(f'Data:      {DATA_DIR}')
print(f'Container: {CONTAINER}')
print(f'Script:    {SCRIPT}')

---
## 2. Select PDB inputs

In [ ]:
### CHOOSE PDB FILES ###

# >> Option A: single file
pdb_files = [
    f'{DATA_DIR}ZAPP_i1_P1D1__bestHIT_active_site_wBB_wKCX__netCHG_0.pdb',
]

# >> Option B: all PDBs in data dir
# pdb_files = sorted(glob.glob(f'{DATA_DIR}*.pdb'))

# >> Option C: specific subset
# pdb_files = sorted(glob.glob(f'{DATA_DIR}ZAPP_i1_*.pdb'))

print(f'{len(pdb_files)} PDB file(s):')
for f in pdb_files:
    print(f'  {os.path.basename(f)}')

---
## 3. Pipeline parameters

Every parameter here maps 1:1 to a CLI flag in `run_neb_ts.py`.

In [ ]:
# ┌─────────────────────────────────────────────────────┐
# │  MODE                                               │
# │  'quick'    = NEB only (fastest, rough barriers)    │
# │  'standard' = NEB + CI-NEB (good TS estimate)      │
# │  'full'     = NEB + CI-NEB + Sella + freq (best)   │
# └─────────────────────────────────────────────────────┘
mode = 'standard'

# ┌─────────────────────────────────────────────────────┐
# │  MACE MODEL                                        │
# │  'mace-omol' = charge-aware, trained on TS data    │
# │                (RECOMMENDED but needs A6000/L40)    │
# │  'mace-mp'   = r2SCAN (all elements, A4000 OK)     │
# │  or: direct path to .model file                    │
# └─────────────────────────────────────────────────────┘
model = 'mace-mp'

# ┌─────────────────────────────────────────────────────┐
# │  CONSTRAINT MODE                                   │
# │  'ca-only'        CA fixed, everything else free   │
# │                   (sidechains, waters, backbone     │
# │                    C/N/O all move). DEFAULT.        │
# │  'backbone'       CA/C/N/O fixed during opt,       │
# │                   CA-only during MD. Waters free.   │
# │  'backbone-water' Like backbone + pin water O.     │
# │  'ca-restrained'  CA fixed + soft restraints on    │
# │                   isolated residue termini.         │
# │  'none'           No constraints.                  │
# └─────────────────────────────────────────────────────┘
constraint_mode = 'ca-only'

# ┌─────────────────────────────────────────────────────┐
# │  NEB PARAMETERS                                    │
# └─────────────────────────────────────────────────────┘
n_images      = 15     # NEB images (odd; 15 gives good resolution)
k_spring      = 1.0    # spring constant between images (0.5-1.5)
fmax_neb_nc   = 0.40   # convergence for no-climb phase (eV/A)
steps_noclimb = 200    # max optimizer steps (no-climb)
fmax_neb_cl   = 0.045  # convergence for climbing image
steps_climb   = 250    # max optimizer steps (climbing)

# ┌─────────────────────────────────────────────────────┐
# │  SELLA TS REFINEMENT (only used in 'full' mode)    │
# │  WARNING: crashes on large systems (>200 free atoms)│
# │  Use 'standard' mode instead for >500 atom systems │
# └─────────────────────────────────────────────────────┘
fmax_sella  = 0.02    # saddle point convergence
steps_sella = 1000    # max Sella steps

# ┌─────────────────────────────────────────────────────┐
# │  ENDPOINT GENERATION                               │
# └─────────────────────────────────────────────────────┘
fmax_end_spring = 0.10
fmax_end_final  = 0.04

# ┌─────────────────────────────────────────────────────┐
# │  BOND-BREAKING SPRINGS (k=3 optimal from testing)  │
# └─────────────────────────────────────────────────────┘
spring_k    = 3.0
spring_fmax = 3.0

# ┌─────────────────────────────────────────────────────┐
# │  MD EQUILIBRATION (200 steps optimal from testing) │
# └─────────────────────────────────────────────────────┘
md_steps = 200
md_temp  = 300.0

# ┌─────────────────────────────────────────────────────┐
# │  FLAGS                                             │
# └─────────────────────────────────────────────────────┘
skip_freq   = False
skip_sella  = False
force_sella = False
pre_relax   = False  # True for docked/chimeric inputs with clashes

print(f'Mode: {mode} | Model: {model} | Constraints: {constraint_mode} | Images: {n_images}')

---
## 4. SLURM / GPU configuration

In [ ]:
### GPU / QUEUE ###
# ┌──────────────────────────────────────────────────────────────┐
# │  GPU MEMORY REQUIREMENTS (approximate, 890-atom system):    │
# │                                                              │
# │  Model          │ VRAM needed │ GPU options                  │
# │  mace-mp        │ ~8 GB       │ a4000 (16 GB) OK             │
# │  mace-omol (XL) │ ~20 GB      │ a4000 OOM! use b4000 / h200  │
# │  mace-mh        │ ~10 GB      │ a4000 OK                     │
# └──────────────────────────────────────────────────────────────┘

gpu_type   = 'h200'        # 'a4000' (16GB), 'b4000' (32GB), 'h200' (80GB)
partition  = 'gpu-bf'      # 'gpu', 'gpu-bf', 'gpu-b4000'
memory     = '32g'
time_limit = '03:59:00'    # HH:MM:SS
cpus       = 8

# Auto-suggest GPU based on model
if model == 'mace-omol' and gpu_type == 'a4000':
    print('WARNING: mace-omol (extra_large) needs >16 GB VRAM. Switching to h200.')
    gpu_type = 'h200'
    partition = 'gpu-bf'
    memory = '32g'

print(f'GPU: {gpu_type} | Partition: {partition} | Mem: {memory} | Time: {time_limit}')

---
## 5. Build commands

In [ ]:
### BUILD COMMANDS ###
commands = []

# apptainer prefix with GPU support + bind mounts + PYTHONPATH for sella
apptainer_prefix = (
    f'apptainer exec --nv'
    f' --bind /home:/home --bind /mnt:/mnt --bind /net:/net'
    f' --env "PYTHONPATH={LOCAL_PKGS}"'
    f' {CONTAINER}'
    f' python'
)

for pdb_path in pdb_files:
    parts = [
        apptainer_prefix,
        SCRIPT,
        pdb_path,
        f'--model {model}',
        f'--mode {mode}',
        f'--constraint-mode {constraint_mode}',
        f'--n-images {n_images}',
        f'--k-spring {k_spring}',
        f'--fmax-neb-noclimb {fmax_neb_nc}',
        f'--steps-noclimb {steps_noclimb}',
        f'--fmax-neb-climb {fmax_neb_cl}',
        f'--steps-climb {steps_climb}',
        f'--fmax-sella {fmax_sella}',
        f'--steps-sella {steps_sella}',
        f'--fmax-end-spring {fmax_end_spring}',
        f'--fmax-end-final {fmax_end_final}',
        f'--spring-k {spring_k}',
        f'--spring-fmax {spring_fmax}',
        f'--md-steps {md_steps}',
        f'--md-temp {md_temp}',
    ]
    if skip_freq:   parts.append('--skip-freq')
    if skip_sella:  parts.append('--skip-sella')
    if force_sella: parts.append('--force-sella')
    if pre_relax:   parts.append('--pre-relax')
    
    commands.append(' '.join(parts))

print(f'{len(commands)} command(s) built.\n')
for i, c in enumerate(commands):
    print(f'--- CMD {i+1} ---')
    print(c.replace(' --', ' \\\n    --'))
    print()

---
## 6a. Interactive GPU test commands

Copy-paste into a `gpu_16g` interactive session.

In [ ]:
print('='*60)
print(' INTERACTIVE GPU COMMANDS')
print('='*60)
print()
print('# 1. Get a GPU')
print(f'gpu_16g')
print()
print('# 2. cd to project')
print(f'cd {PROJECT_DIR}')
print()
print('# 3. Run')
for i, cmd in enumerate(commands):
    if len(commands) > 1:
        print(f'\n# --- Job {i+1} ---')
    print(cmd)
print()
print('# 4. Exit GPU when done')
print('exit')

---
## 6b. Write commands file & sbatch submit script

In [ ]:
### WRITE FILES ###
job_name = f'NEB_TS_{mode}_{model}'

# >> commands file
cmds_file = os.path.join(CMDS_DIR, job_name)
with open(cmds_file, 'w') as f:
    f.write('\n'.join(commands) + '\n')

# >> submit script
submit_file = os.path.join(SUBMIT_DIR, f'{job_name}.sh')
n_jobs = len(commands)

if n_jobs == 1:
    script_body = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={partition}
#SBATCH --gres=gpu:{gpu_type}:1
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}
#SBATCH --time={time_limit}
#SBATCH --output={LOGS_DIR}{job_name}_%j.stdout
#SBATCH --error={LOGS_DIR}{job_name}_%j.stderr

cd {PROJECT_DIR}
echo "Job started: $(date) | Node: $(hostname) | GPU: $(nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null)"

{commands[0]}

echo "Job finished: $(date)"
"""
else:
    script_body = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={partition}
#SBATCH --gres=gpu:{gpu_type}:1
#SBATCH --mem={memory}
#SBATCH --cpus-per-task={cpus}
#SBATCH --time={time_limit}
#SBATCH --array=1-{n_jobs}
#SBATCH --output={LOGS_DIR}{job_name}_%A_%a.stdout
#SBATCH --error={LOGS_DIR}{job_name}_%A_%a.stderr

cd {PROJECT_DIR}
CMD=$(sed -n "${{SLURM_ARRAY_TASK_ID}}p" {cmds_file})
echo "Job ${{SLURM_ARRAY_TASK_ID}}/{n_jobs} started: $(date) | Node: $(hostname)"
$CMD
echo "Job ${{SLURM_ARRAY_TASK_ID}} finished: $(date)"
"""

with open(submit_file, 'w') as f:
    f.write(script_body)
os.chmod(submit_file, os.stat(submit_file).st_mode | stat.S_IEXEC)

# >> print summary
array_tag = f' | array 1-{n_jobs}' if n_jobs > 1 else ''
print('='*70)
print(f'  NEB-TS Job Summary | {n_jobs} cmd(s) | mode={mode} | {gpu_type}{array_tag}')
print('='*70)
print(f'| [ CMDS ]: {cmds_file}')
print(f'| [ LOGS ]: {LOGS_DIR}{job_name}_*.stdout')
print('='*70)
print(f'| [SUBMIT]:')
print(f'sbatch {submit_file}')
print('='*70)

---
## 7. Check outputs

In [ ]:
import json

OUTPUT_DIR = os.path.join(PROJECT_DIR, 'outputs/')
output_dirs = sorted(glob.glob(f'{OUTPUT_DIR}*/')) if os.path.isdir(OUTPUT_DIR) else []

if not output_dirs:
    print('No outputs yet.')
else:
    print(f'{len(output_dirs)} output dir(s):\n')
    for d in output_dirs:
        name = os.path.basename(d.rstrip('/'))
        summary_f = os.path.join(d, 'summary.json')
        
        steps_done = []
        if os.path.isfile(os.path.join(d, 'relax', 'relaxation-start.xyz')): steps_done.append('relax-start')
        if os.path.isfile(os.path.join(d, 'relax', 'relaxation-end.xyz')):   steps_done.append('relax-end')
        if os.path.isfile(os.path.join(d, 'ts', 'path-neb-climb.xyz')):      steps_done.append('NEB')
        if os.path.isfile(os.path.join(d, 'ts', 'path-after-sella-ts.xyz')): steps_done.append('Sella')
        if os.path.isfile(os.path.join(d, 'ts', 'ts-neb-highest.xyz')):      steps_done.append('TS(NEB)')
        
        if os.path.isfile(summary_f):
            s = json.load(open(summary_f))
            print(f'  {name}')
            print(f'    mode={s.get("mode","?")} | barrier={s.get("barrier_fwd_kcal",0):.1f} kcal/mol | '
                  f'time={s.get("elapsed_min",0):.1f} min | {" > ".join(steps_done)}')
        else:
            print(f'  {name}  [in progress: {" > ".join(steps_done) if steps_done else "not started"}]')

---
## Reference

### Modes
| Mode | Steps | Sella? | Freq? | Use case |
|------|-------|--------|-------|----------|
| `quick` | relax + NEB (no climb) | no | no | Fast screening of many theozymes |
| `standard` | relax + NEB + CI-NEB | no | no | Default. Good TS barrier estimate |
| `full` | relax + NEB + CI-NEB + Sella + freq | yes | yes | Publication. Validated saddle point |

### Adding new ligand types
Edit `BOND_BREAKING_DEFS` in `run_neb_ts.py`:
```python
BOND_BREAKING_DEFS['NEW_LIG'] = [
    ('atom_forming', 'atom_nuc', 1.4, 'attractive'),  # new bond
    ('atom_breaking', 'atom_lg', 2.1, 'repulsive'),    # old bond
]
```

### Environment
- **Container:** `/net/software/containers/universal.sif` (mace 0.3.15 + e3nn 0.4.4 + torch 2.11)
- **Sella + JAX:** from `.local_pkgs/` overlay via `PYTHONPATH`
- **No conda/venv dependency** — runs entirely through apptainer

### Key output files
```
outputs/<system>-<model>/
  relax/relaxation-start.xyz     # optimized reactant
  relax/relaxation-end.xyz       # optimized product
  ts/path-neb-climb.xyz          # full NEB path (all images)
  ts/path-after-sella-ts.xyz     # Sella-refined TS (full mode)
  ts/ts-neb-highest.xyz          # CI-NEB TS (standard mode)
  ts/energy_profile.png          # energy diagram
  summary.json                   # barriers, timings, metadata
```